In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/luongdiem/course-llm-embeddings/llm_profile_no_prereq_followup_embeddings.npy
/kaggle/input/datasets/luongdiem/course-llm-embeddings/embeddings_course_index.json
/kaggle/input/datasets/luongdiem/course-llm-embeddings/course_dup_groups.parquet
/kaggle/input/datasets/luongdiem/course-llm-embeddings/raw_course_text_embeddings.npy
/kaggle/input/datasets/luongdiem/course-llm-embeddings/llm_profile_embeddings.npy
/kaggle/input/datasets/luongdiem/course-llm-embeddings/raw_embeddings_course_index.json
/kaggle/input/datasets/luongdiem/evaluator/evaluator.py
/kaggle/input/datasets/luongdiem/lightgcn-backbone-cold/user_index_cold.json
/kaggle/input/datasets/luongdiem/lightgcn-backbone-cold/g_u.npy
/kaggle/input/datasets/luongdiem/lightgcn-backbone-cold/g_i.npy
/kaggle/input/datasets/luongdiem/lightgcn-backbone-cold/item_index.json
/kaggle/input/datasets/luongdiem/cold-scenario/test_cold.parquet
/kaggle/input/datasets/luongdiem/cold-scenario/val_cold.parquet
/kaggle/input/da

In [2]:
%%writefile coldrec_adapter.py
import copy
import hashlib
import json
import os
import pickle
import sys
import time
import numpy as np
import pandas as pd
import torch

ITEM_IDX_PATH = "/kaggle/input/datasets/luongdiem/lightgcn-backbone-cold/item_index.json"
USER_IDX_PATH = "/kaggle/input/datasets/luongdiem/lightgcn-backbone-cold/user_index_cold.json"
GU_PATH = "/kaggle/input/datasets/luongdiem/lightgcn-backbone-cold/g_u.npy"
GI_PATH = "/kaggle/input/datasets/luongdiem/lightgcn-backbone-cold/g_i.npy"

RAW_IDX_PATH = "/kaggle/input/datasets/luongdiem/course-llm-embeddings/raw_embeddings_course_index.json"
LLM_IDX_PATH = "/kaggle/input/datasets/luongdiem/course-llm-embeddings/embeddings_course_index.json"
RAW_NPY = "/kaggle/input/datasets/luongdiem/course-llm-embeddings/raw_course_text_embeddings.npy"
LLM_NPY = "/kaggle/input/datasets/luongdiem/course-llm-embeddings/llm_profile_embeddings.npy"
LLM_3D_NPY = "/kaggle/input/datasets/luongdiem/course-llm-embeddings/llm_profile_no_prereq_followup_embeddings.npy"

COLD_VAL_JSON = "/kaggle/input/datasets/luongdiem/cold-scenario/cold_items_val.json"
COLD_TEST_JSON = "/kaggle/input/datasets/luongdiem/cold-scenario/cold_items_test.json"
TRAIN_PARQUET = "/kaggle/input/datasets/luongdiem/cold-scenario/train_cold.parquet"
VAL_PARQUET = "/kaggle/input/datasets/luongdiem/cold-scenario/val_cold.parquet"
TEST_PARQUET = "/kaggle/input/datasets/luongdiem/cold-scenario/test_cold.parquet"

OUTPUT_DIR = "/kaggle/working"


def find_file_in_kaggle(filename, default_path=None):
    """Tự động tìm đường dẫn tệp trong /kaggle/input hoặc /kaggle/working."""
    if default_path and os.path.exists(default_path):
        return default_path
    for base_dir in ("/kaggle/input", "/kaggle/working", "."):
        if not os.path.exists(base_dir):
            continue
        for root, _, files in os.walk(base_dir):
            if filename in files:
                return os.path.join(root, filename)
    raise FileNotFoundError(
        f"Không tìm thấy tệp '{filename}' trong /kaggle/input! Hãy kiểm tra"
        " Dataset đã gắn vào Notebook."
    )


# Đường dẫn tệp phân nhóm trùng lặp P-RQ3 (tự động tìm nếu đặt ở thư mục khác)
DUP_GROUPS_PARQUET = find_file_in_kaggle(
    "course_dup_groups.parquet",
    default_path="/kaggle/input/datasets/luongdiem/course-llm-embeddings/course_dup_groups.parquet",
)

# Tự động tìm và import evaluator.py (hoặc evaluate.py)
try:
    eval_file_path = find_file_in_kaggle("evaluator.py")
    sys.path.insert(0, os.path.dirname(eval_file_path))
    from evaluator import evaluate, topk_random_ties
except FileNotFoundError:
    eval_file_path = find_file_in_kaggle("evaluate.py")
    sys.path.insert(0, os.path.dirname(eval_file_path))
    from evaluate import evaluate, topk_random_ties


def md5_hash(filepath):
    """Tính mã băm MD5 của tệp để truy vết phiên bản"""
    return hashlib.md5(open(filepath, "rb").read()).hexdigest()


def to_bool_mask(ids, c2i, n_items=3769):
    """Chuyển danh sách course_id sang mặt nạ bool dài 3.769"""
    m = torch.zeros(n_items, dtype=torch.bool)
    m[[c2i[str(c)] for c in ids]] = True
    return m


def build_hist(df, u2i, c2i):
    """Dựng từ điển lịch sử tương tác {u_idx: [i_idx, ...]}."""
    u_mapped = df["user_id"].astype(str).map(u2i)
    i_mapped = df["course_id"].astype(str).map(c2i)
    temp = pd.DataFrame({"u": u_mapped, "i": i_mapped}).dropna()
    temp["u"] = temp["u"].astype(int)
    temp["i"] = temp["i"].astype(int)
    return temp.groupby("u")["i"].apply(list).to_dict()


def _extract_id_list(json_path, key_name):
    data = json.load(open(json_path))
    items = data[key_name] if isinstance(data, dict) else data
    return [str(x) for x in items]


# Lớp bộ chuyển đổi dữ liệu và kiểm tra giao thức
class ColdRecDataAdapter:

    def __init__(self, output_dir=OUTPUT_DIR):
        self.output_dir = output_dir
        os.makedirs(self.output_dir, exist_ok=True)
        self._load_and_verify_all()

    def _load_and_verify_all(self):
        print("=== KIỂM TRA MD5 VÀ BẢNG ÁNH XẠ CHỈ SỐ CHUNG ===")
        # Đọc và đối chiếu bảng ánh xạ chỉ số chung 
        self.course_ids = _extract_id_list(ITEM_IDX_PATH, "course_id")
        self.users = _extract_id_list(USER_IDX_PATH, "user_id")

        assert len(self.course_ids) == 3769 and self.course_ids == sorted(self.course_ids), "Lỗi bảng item_index.json!"
        assert len(self.users) == 98220 and self.users == sorted(self.users), "Lỗi bảng user_index_cold.json!"

        if os.path.exists(RAW_IDX_PATH):
            raw_cids = _extract_id_list(RAW_IDX_PATH, "course_id")
            assert self.course_ids == raw_cids, "item_index.json không khớp raw_embeddings_course_index.json!"
        if os.path.exists(LLM_IDX_PATH):
            llm_cids = _extract_id_list(LLM_IDX_PATH, "course_id")
            assert self.course_ids == llm_cids, "item_index.json không khớp embeddings_course_index.json!"

        self.item2idx = {c: i for i, c in enumerate(self.course_ids)}
        self.user2idx = {u: i for i, u in enumerate(self.users)}
        self.n_users = len(self.users)
        self.n_items = len(self.course_ids)

        # Đọc danh sách khóa lạnh (251 cold-val, 502 cold-test)
        self.cold_val_ids = [str(c) for c in json.load(open(COLD_VAL_JSON))]
        self.cold_test_ids = [str(c) for c in json.load(open(COLD_TEST_JSON))]
        assert len(self.cold_val_ids) == 251 and len(self.cold_test_ids) == 502

        self.cold_val_idx = sorted([self.item2idx[c] for c in self.cold_val_ids])
        self.cold_test_idx = sorted([self.item2idx[c] for c in self.cold_test_ids])
        self.cold_idx = np.array(sorted(self.cold_val_idx + self.cold_test_idx), dtype=np.int64)
        assert len(self.cold_idx) == 753

        # Đọc các tệp tương tác và chạy assert chống rò rỉ
        self.tr_cold = pd.read_parquet(TRAIN_PARQUET)
        self.va_cold = pd.read_parquet(VAL_PARQUET)
        self.te_cold = pd.read_parquet(TEST_PARQUET)

        tr_items = set(self.tr_cold["course_id"].astype(str))
        cv, ct = set(self.cold_val_ids), set(self.cold_test_ids)
        assert not (cv & tr_items) and not (ct & tr_items) and not (cv & ct), "Rò rỉ khóa lạnh vào train hoặc trùng val/test!"
        assert set(self.va_cold["course_id"].astype(str)) <= cv
        assert set(self.te_cold["course_id"].astype(str)) <= ct
        assert set(self.va_cold["user_id"].astype(str)) <= set(self.tr_cold["user_id"].astype(str))
        assert set(self.te_cold["user_id"].astype(str)) <= set(self.tr_cold["user_id"].astype(str))

        # Xác định đúng 2.401 khóa ấm có tương tác trong train_cold
        self.warm_active_idx = np.unique(
            self.tr_cold["course_id"]
            .astype(str)
            .map(self.item2idx)
            .to_numpy(dtype=np.int64)
        )
        assert len(self.warm_active_idx) == 2401 and not (set(self.warm_active_idx.tolist()) & set(self.cold_idx.tolist()))

        # Đọc và kiểm tra Backbone LightGCN (g_u, g_i)
        self.g_u = np.load(GU_PATH).astype(np.float32)
        self.g_i = np.load(GI_PATH).astype(np.float32)

        gu_md5, gi_md5 = md5_hash(GU_PATH), md5_hash(GI_PATH)
        print(f"MD5 g_u.npy: {gu_md5} (Kỳ vọng: 7a170558...)")
        print(f"MD5 g_i.npy: {gi_md5} (Kỳ vọng: d17ad9df...)")

        assert self.g_u.shape == (98220, 64) and self.g_i.shape == (3769, 64)
        assert np.allclose(self.g_i[self.cold_idx], 0.0), "Khóa lạnh (753 khóa) phải có g_i = 0!"

        # Đảm bảo 615 khóa ấm không có tương tác trong train_cold cũng có g_i = 0
        inactive_mask = np.ones(self.n_items, dtype=bool)
        inactive_mask[self.warm_active_idx] = False
        assert np.allclose(self.g_i[inactive_mask], 0.0), "615 khóa ấm không tương tác phải có g_i = 0!"

        # Đọc và chuẩn hóa L2 cho các biến thể đặc trưng nội dung (raw, llm, fused, llm_3d, fused_3d)
        print(f"MD5 raw_emb: {md5_hash(RAW_NPY)} | MD5 llm_emb: {md5_hash(LLM_NPY)}")

        nz = lambda X: X / np.linalg.norm(X, axis=1, keepdims=True).clip(1e-9)
        self.S_raw = nz(np.load(RAW_NPY).astype(np.float32))
        self.S_llm = nz(np.load(LLM_NPY).astype(np.float32))
        assert self.S_raw.shape == (3769, 384) and self.S_llm.shape == (3769,384,)
        self.S_fused = np.concatenate([self.S_raw, self.S_llm], axis=1).astype(np.float32)  # [3769, 768]

        self.contents = {
            "raw": self.S_raw,
            "llm": self.S_llm,
            "fused": self.S_fused,
        }

        if os.path.exists(LLM_3D_NPY):
            print(f"MD5 llm_3d_emb: {md5_hash(LLM_3D_NPY)}")
            self.S_llm_3d = nz(np.load(LLM_3D_NPY).astype(np.float32))
            assert self.S_llm_3d.shape == (3769, 384)
            self.S_fused_3d = np.concatenate(
                [self.S_raw, self.S_llm_3d], axis=1
            ).astype(np.float32)
            self.contents["llm_3d"] = self.S_llm_3d
            self.contents["fused_3d"] = self.S_fused_3d

        # Dựng từ điển mặt nạ lịch sử, đích đánh giá và mặt nạ ứng viên (a), (b)
        self.hist_tr = build_hist(self.tr_cold, self.user2idx, self.item2idx)
        self.hist_trva = build_hist(
            pd.concat([self.tr_cold, self.va_cold], ignore_index=True),
            self.user2idx,
            self.item2idx,
        )
        self.val_target_dict = build_hist(
            self.va_cold, self.user2idx, self.item2idx
        )
        self.test_target_dict = build_hist(
            self.te_cold, self.user2idx, self.item2idx
        )
        assert (
            len(self.val_target_dict) == 2092
            and len(self.test_target_dict) == 3340
        )

        self.cand_val_a = to_bool_mask(
            self.cold_val_ids, self.item2idx, self.n_items
        )
        self.cand_test_a = to_bool_mask(
            self.cold_test_ids, self.item2idx, self.n_items
        )
        self.cand_test_b = torch.ones(self.n_items, dtype=torch.bool)

        # Phân nhóm P-RQ3 ("trung" vs "moi") từ course_dup_groups.parquet
        dup_df = pd.read_parquet(DUP_GROUPS_PARQUET)
        c2dup = dict(zip(dup_df["course_id"].astype(str), dup_df["dup_group_id"]))
        active_warm_course_ids = [self.course_ids[i] for i in self.warm_active_idx]
        warm_active_dup_groups = {c2dup[c] for c in active_warm_course_ids}

        self.cold_test_dup_ids = [c for c in self.cold_test_ids if c2dup[c] in warm_active_dup_groups]
        self.cold_test_new_ids = [
            c
            for c in self.cold_test_ids
            if c2dup[c] not in warm_active_dup_groups
        ]
        assert (
            len(self.cold_test_dup_ids) == 124
            and len(self.cold_test_new_ids) == 378
        ), (
            f"Sai số lượng khóa trùng/mới: {len(self.cold_test_dup_ids)} /"
            f" {len(self.cold_test_new_ids)}"
        )

        self.groups_prq3 = {
            "trung": {self.item2idx[c] for c in self.cold_test_dup_ids},
            "moi": {self.item2idx[c] for c in self.cold_test_new_ids},
        }
        print("=== HOÀN TẤT KIỂM TRA: MỌI ASSERT ĐỀU ĐẠT CHUẨN! ===\n")

    def export_coldrec_files(self, dataset_name="MOOCCubeX", content_variant="raw"):
        """Xuất bộ dữ liệu sang đúng cấu trúc thư mục mà ColdRec (ALDI, CGRC, USIM)
        và mã chính thức zfnWong/ALDI yêu cầu, giữ nguyên chỉ số 0..98219 và
        0..3768.
        """
        ds_dir = os.path.join(self.output_dir, "ColdRec_Data", dataset_name)
        cold_item_dir = os.path.join(ds_dir, "cold_item")
        os.makedirs(ds_dir, exist_ok=True)
        os.makedirs(cold_item_dir, exist_ok=True)

        # Ánh xạ các bảng tương tác sang chỉ số nguyên (user, item)
        def to_mapped_df(df):
            u = df["user_id"].astype(str).map(self.user2idx).astype(np.int64)
            i = df["course_id"].astype(str).map(self.item2idx).astype(np.int64)
            return pd.DataFrame({"user": u, "item": i})

        tr_df = to_mapped_df(self.tr_cold)
        va_df = to_mapped_df(self.va_cold)
        te_df = to_mapped_df(self.te_cold)
        cold_all_df = pd.concat([va_df, te_df], ignore_index=True)

        # Tạo warm_val, warm_test, overall_val, overall_test theo đúng cấu trúc ColdRec/data/convert.py
        # (lấy 1 tương tác ấm/học viên từ tập train để các hàm đánh giá warm/overall nội bộ của ColdRec không lỗi chia 0)
        warm_va_df = tr_df[tr_df["user"].isin(set(va_df["user"]))].drop_duplicates(
            subset=["user"], keep="first"
        )
        warm_te_df = tr_df[tr_df["user"].isin(set(te_df["user"]))].drop_duplicates(
            subset=["user"], keep="first"
        )
        overall_va_df = pd.concat([va_df, warm_va_df], ignore_index=True)
        overall_te_df = pd.concat([te_df, warm_te_df], ignore_index=True)

        # Lưu đủ 8 tệp CSV vào cả ds_dir và ds_dir/cold_item/
        csv_files = {
            "warm_train.csv": tr_df,
            "cold_item_val.csv": va_df,
            "cold_item_test.csv": te_df,
            "cold_item.csv": cold_all_df,
            "warm_val.csv": warm_va_df,
            "warm_test.csv": warm_te_df,
            "overall_val.csv": overall_va_df,
            "overall_test.csv": overall_te_df,
        }
        for fname, df_out in csv_files.items():
            df_out.to_csv(os.path.join(ds_dir, fname), index=False)
            df_out.to_csv(os.path.join(cold_item_dir, fname), index=False)

        # Lưu danh sách ID khóa ấm có tương tác (2.401 khóa) và khóa lạnh
        np.save(os.path.join(ds_dir, "warm_item_ids.npy"), self.warm_active_idx)
        np.save(os.path.join(ds_dir, "cold_val_item_ids.npy"),np.array(self.cold_val_idx, dtype=np.int64),)
        np.save(os.path.join(ds_dir, "cold_test_item_ids.npy"),np.array(self.cold_test_idx, dtype=np.int64),)
        np.save(os.path.join(ds_dir, "cold_item_ids.npy"), self.cold_idx)

        # Lưu Pre-trained Teacher Embeddings (LightGCN nền chung) dạng .npy và .pt
        warm_emb_concat = np.concatenate([self.g_u, self.g_i], axis=0)  # [98220 + 3769, 64]
        np.save(os.path.join(ds_dir, f"{dataset_name}_lightgcn.npy"), warm_emb_concat)
        np.save(os.path.join(ds_dir, "warm_emb.npy"), warm_emb_concat)
        np.save(os.path.join(ds_dir, "g_u.npy"), self.g_u)
        np.save(os.path.join(ds_dir, "g_i.npy"), self.g_i)

        torch.save(torch.from_numpy(self.g_u), os.path.join(ds_dir, "g_u_tensor.pt"))
        torch.save(torch.from_numpy(self.g_i), os.path.join(ds_dir, "g_i_tensor.pt"))
        torch.save({"user_emb": torch.from_numpy(self.g_u), "item_emb": torch.from_numpy(self.g_i)},os.path.join(ds_dir, "warm_emb_dict.pt"),)

        # Lưu ma trận đặc trưng nội dung khóa học theo biến thể được chọn ('raw', 'llm', 'fused', ...)
        content_mat = self.contents[content_variant]
        np.save(os.path.join(ds_dir, f"{dataset_name}_item_content.npy"), content_mat)
        np.save(os.path.join(ds_dir, "item_content.npy"), content_mat)

        # Chuẩn bị các mảng np.int32 theo đúng yêu cầu của ColdRec/main.py và ColdRec/data/convert.py
        warm_user_arr = np.array(sorted(set(tr_df["user"].tolist())), dtype=np.int32)
        warm_item_arr = np.array(self.warm_active_idx, dtype=np.int32)
        cold_user_arr = np.array(sorted(set(cold_all_df["user"].tolist())), dtype=np.int32)
        cold_item_arr = np.array(self.cold_idx, dtype=np.int32)

        info_dict = {
            # Các khóa bắt buộc cho ColdRec/main.py và ColdRec/data/convert.py
            "user_num": self.n_users,
            "item_num": self.n_items,
            "user_array": np.arange(self.n_users, dtype=np.int32),
            "item_array": np.arange(self.n_items, dtype=np.int32),
            "warm_user": warm_user_arr,
            "warm_item": warm_item_arr,
            "cold_user": cold_user_arr,
            "cold_item": cold_item_arr,
            "train_user": warm_user_arr,
            "warm_val_user": np.array(
                sorted(set(warm_va_df["user"].tolist())), dtype=np.int32
            ),
            "warm_test_user": np.array(
                sorted(set(warm_te_df["user"].tolist())), dtype=np.int32
            ),
            "cold_val_user": np.array(
                sorted(set(va_df["user"].tolist())), dtype=np.int32
            ),
            "cold_test_user": np.array(
                sorted(set(te_df["user"].tolist())), dtype=np.int32
            ),
            "overall_val_user": np.array(
                sorted(set(overall_va_df["user"].tolist())), dtype=np.int32
            ),
            "overall_test_user": np.array(
                sorted(set(overall_te_df["user"].tolist())), dtype=np.int32
            ),
            "train_item": warm_item_arr,
            "warm_val_item": np.array(
                sorted(set(warm_va_df["item"].tolist())), dtype=np.int32
            ),
            "warm_test_item": np.array(
                sorted(set(warm_te_df["item"].tolist())), dtype=np.int32
            ),
            "cold_val_item": np.array(self.cold_val_idx, dtype=np.int32),
            "cold_test_item": np.array(self.cold_test_idx, dtype=np.int32),
            "overall_val_item": np.array(
                sorted(set(overall_va_df["item"].tolist())), dtype=np.int32
            ),
            "overall_test_item": np.array(
                sorted(set(overall_te_df["item"].tolist())), dtype=np.int32
            ),
            # Các khóa bổ sung cho adapter nội bộ / ALDI
            "warm_items": set(self.warm_active_idx.tolist()),
            "cold_val_items": set(self.cold_val_idx),
            "cold_test_items": set(self.cold_test_idx),
            "cold_items": set(self.cold_idx.tolist()),
            "emb_dim": 64,
            "content_dim": int(content_mat.shape[1]),
            "user_his": self.hist_tr,
            "val_user_his": self.hist_trva,
        }

        # Lưu cả info_dict.pkl, para_dict.pkl và n_user_item.pkl ở cả 2 thư mục
        for target_d in (ds_dir, cold_item_dir):
            with open(os.path.join(target_d, "info_dict.pkl"), "wb") as f:
                pickle.dump(info_dict, f, protocol=4)
            with open(os.path.join(target_d, "para_dict.pkl"), "wb") as f:
                pickle.dump(info_dict, f, protocol=4)
            with open(os.path.join(target_d, "n_user_item.pkl"), "wb") as f:
                pickle.dump({"user": self.n_users, "item": self.n_items}, f, protocol=4)

        print(
            f"[OK] Đã xuất dữ liệu ColdRec ({content_variant},"
            f" dim={content_mat.shape[1]}) tại: {ds_dir}"
        )
        return ds_dir


# Các hàm đánh giá của nhóm (tập a, b, P-RQ3, P-RQ4)
@torch.no_grad()
def eval_subgroups(
    score_fn, targets, mask, cand, groups, ks=(5, 10, 20), tie_seed=0, bs=1024
):
    """Hàm tính P-RQ3 đúng chuẩn:

    Xếp hạng trên TOÀN BỘ 502 khóa của tập (a), sau đó chỉ tính chỉ số trên đích
    thuộc từng nhóm.
    """
    users, K = sorted(targets), max(ks)
    g = torch.Generator().manual_seed(tie_seed)
    disc = 1.0 / np.log2(np.arange(2, K + 2))
    cand = torch.as_tensor(cand)
    out = {
        n: {f"{m}@{k}": [] for m in ("Recall", "NDCG", "HR") for k in ks}
        for n in groups
    }
    for s in range(0, len(users), bs):
        ub = users[s : s + bs]
        sc = score_fn(torch.tensor(ub)).float().cpu().clone()
        sc[:, ~cand] = -float("inf")
        for r, u in enumerate(ub):
            if mask.get(u):
                sc[r, list(mask[u])] = -float("inf")
        top = topk_random_ties(sc, K, g)
        for r, u in enumerate(ub):
            for n, gset in groups.items():
                gt = [i for i in targets[u] if i in gset]
                if not gt:
                    continue
                hit = np.isin(top[r], gt).astype(float)
                for k in ks:
                    h = hit[:k]
                    idcg = disc[: min(len(gt), k)].sum()
                    out[n][f"Recall@{k}"].append(h.sum() / len(gt))
                    out[n][f"NDCG@{k}"].append((h * disc[:k]).sum() / idcg)
                    out[n][f"HR@{k}"].append(float(h.sum() > 0))
    return {n: {m: np.array(v) for m, v in d.items()} for n, d in out.items()}


@torch.no_grad()
def evaluate_coldrec_embeddings(
    adapter, u_emb, i_emb, model_name, variant_name, seed, device="cuda:0" if torch.cuda.is_available() else "cpu"
):
    """Hàm cầu nối bắt buộc:

    Nhận u_emb [98220, 64] và i_emb [3769, 64] xuất ra từ mô hình ColdRec (ALDI
    / CGRC / USIM), chấm điểm bằng evaluator.py và lưu đầy đủ file phục
    vụ Wilcoxon + P-RQ3.
    """
    u_t = torch.as_tensor(u_emb, dtype=torch.float32, device=device)
    i_t = torch.as_tensor(i_emb, dtype=torch.float32, device=device)
    assert u_t.shape == (98220, 64) and i_t.shape == (3769, 64)

    score_fn = lambda ub: (u_t[ub.to(device)] @ i_t.t()).cpu()

    # Đánh giá trên tập (a) (502 khóa cold-test) và tập (b) (3.769 khóa)
    kw = dict(mask=adapter.hist_trva, ks=(5, 10, 20), tie_seed=seed)
    res_a = evaluate(score_fn, adapter.test_target_dict, cand=adapter.cand_test_a, **kw)
    res_b = evaluate(score_fn, adapter.test_target_dict, cand=adapter.cand_test_b, **kw)

    # Đánh giá P-RQ3 theo nhóm khóa trùng / mới
    res_prq3 = eval_subgroups(
        score_fn,
        adapter.test_target_dict,
        adapter.hist_trva,
        adapter.cand_test_a,
        adapter.groups_prq3,
        ks=(5, 10, 20),
        tie_seed=seed,
    )
    assert (
        len(res_prq3["trung"]["Recall@20"]) == 1028
        and len(res_prq3["moi"]["Recall@20"]) == 2844
    )

    # Lưu biểu diễn người học và khóa học của seed này
    rep_dir = os.path.join(adapter.output_dir, f"results_{model_name}", "item_reps")
    os.makedirs(rep_dir, exist_ok=True)
    np.save(
        os.path.join(rep_dir, f"{model_name}_{variant_name}_seed{seed}.npy"),
        i_t.cpu().numpy(),
    )
    np.save(
        os.path.join(rep_dir, f"u_e_{variant_name}_seed{seed}.npy"),
        u_t.cpu().numpy(),
    )
    np.save(
        os.path.join(rep_dir, f"v_e_{variant_name}_seed{seed}.npy"),
        i_t.cpu().numpy(),
    )

    return res_a, res_b, res_prq3


if __name__ == "__main__":
    # Khởi tạo Adapter, chạy toàn bộ kiểm tra assert và xuất dữ liệu cho cả 5 biến thể (gồm 3D cho E-RQ3)
    adapter = ColdRecDataAdapter()
    for var in ["raw", "llm", "fused", "llm_3d", "fused_3d"]:
        if var in adapter.contents:
            adapter.export_coldrec_files(dataset_name=f"MOOCCubeX_{var}", content_variant=var)

Writing coldrec_adapter.py


In [3]:
!python coldrec_adapter.py

=== KIỂM TRA MD5 VÀ BẢNG ÁNH XẠ CHỈ SỐ CHUNG ===
MD5 g_u.npy: 7a170558642de73d301970e8ce7f0a86 (Kỳ vọng: 7a170558...)
MD5 g_i.npy: d17ad9dfc4f4d6cb6de915c8649070d7 (Kỳ vọng: d17ad9df...)
MD5 raw_emb: f80445649ef8a8217d048bd3e7ac0ad5 | MD5 llm_emb: fc1819307be227a1544e1be0a4017393
MD5 llm_3d_emb: 1af436095463ad0af3ef1203ec0ebd6c
=== HOÀN TẤT KIỂM TRA: MỌI ASSERT ĐỀU ĐẠT CHUẨN! ===

[OK] Đã xuất dữ liệu ColdRec (raw, dim=384) tại: /kaggle/working/ColdRec_Data/MOOCCubeX_raw
[OK] Đã xuất dữ liệu ColdRec (llm, dim=384) tại: /kaggle/working/ColdRec_Data/MOOCCubeX_llm
[OK] Đã xuất dữ liệu ColdRec (fused, dim=768) tại: /kaggle/working/ColdRec_Data/MOOCCubeX_fused
[OK] Đã xuất dữ liệu ColdRec (llm_3d, dim=384) tại: /kaggle/working/ColdRec_Data/MOOCCubeX_llm_3d
[OK] Đã xuất dữ liệu ColdRec (fused_3d, dim=768) tại: /kaggle/working/ColdRec_Data/MOOCCubeX_fused_3d


In [4]:
# Tải ColdRec và CỐ ĐỊNH đúng commit 1108192 
import os
import subprocess
import sys

COLDREC_ROOT = "/kaggle/working/ColdRec"
COLDREC_COMMIT = "1108192"

if not os.path.isdir(os.path.join(COLDREC_ROOT, ".git")):
    subprocess.run(["git", "clone", "-q", "https://github.com/YuanchenBei/ColdRec.git", COLDREC_ROOT], check=True)
subprocess.run(["git", "-C", COLDREC_ROOT, "checkout", "-q", COLDREC_COMMIT], check=True)
head = subprocess.run(["git", "-C", COLDREC_ROOT, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
assert head.startswith(COLDREC_COMMIT), f"ColdRec phải ở commit {COLDREC_COMMIT}, hiện là {head}"
print("ColdRec commit:", head)   # ghi lại commit hash vào log

# model/KNN.py và model/NCL.py import faiss; main.py import toàn bộ model/ nên bắt buộc phải có
try:
    import faiss  
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu"], check=True)
    import faiss  
print("faiss OK")

ColdRec commit: 1108192665aca2cb94b3aeb5bfed29850874eb1c
faiss OK


In [5]:
# Liên kết dữ liệu đã xuất bởi coldrec_adapter.py vào ColdRec/data/ cho 3 dạng đầu vào: raw, fused, llm.
import os
import pickle
import shutil

import numpy as np

COLDREC_ROOT = "/kaggle/working/ColdRec"
DATASETS = ["MOOCCubeX_raw", "MOOCCubeX_fused", "MOOCCubeX_llm"]
CONTENT_DIM = {"MOOCCubeX_raw": 384, "MOOCCubeX_llm": 384, "MOOCCubeX_fused": 768}
CSVS = ["warm_train", "cold_item_val", "cold_item_test", "warm_val", "warm_test", "overall_val", "overall_test"]

os.makedirs(os.path.join(COLDREC_ROOT, "data"), exist_ok=True)
for ds in DATASETS:
    src = f"/kaggle/working/ColdRec_Data/{ds}"
    dst = os.path.join(COLDREC_ROOT, "data", ds)
    assert os.path.isdir(os.path.join(src, "cold_item")), f"Thiếu {src}/cold_item: chạy lại ô 'python coldrec_adapter.py'"
    if os.path.islink(dst):
        os.unlink(dst)
    elif os.path.isdir(dst):
        shutil.rmtree(dst)
    os.symlink(src, dst)

    missing = [f for f in [f"cold_item/{c}.csv" for c in CSVS] + ["cold_item/info_dict.pkl", f"{ds}_item_content.npy"]
               if not os.path.isfile(os.path.join(dst, f))]
    assert not missing, f"{ds}: thiếu {missing}"
    content = np.load(os.path.join(dst, f"{ds}_item_content.npy"), mmap_mode="r")
    assert content.shape == (3769, CONTENT_DIM[ds]), (ds, content.shape)
    info = pickle.load(open(os.path.join(dst, "cold_item", "info_dict.pkl"), "rb"))
    print(f"[OK] {ds}: content{tuple(content.shape)} | cold_item={len(info['cold_item'])} warm_item={len(info['warm_item'])}")

[OK] MOOCCubeX_raw: content(3769, 384) | cold_item=753 warm_item=2401
[OK] MOOCCubeX_fused: content(3769, 768) | cold_item=753 warm_item=2401
[OK] MOOCCubeX_llm: content(3769, 384) | cold_item=753 warm_item=2401


In [6]:
%%writefile coldrec_fix.py
"""coldrec_fix.py - sửa ColdRec để chạy USIM đúng giao thức 

Chạy MỘT lần (idempotent) sau khi coldrec_adapter.py đã xuất dữ liệu:
    patch_coldrec_source()                       # (1)(2)(3) vá mã nguồn
    write_backbone_and_info_dict(ad)             # (4)(5) backbone LightGCN + info_dict.pkl
    verify_identity_mapping(ds)                  # kiểm tra mã nội bộ == mã gốc
"""
import os
import pickle
import re
import subprocess
import sys

import numpy as np
import torch

COLDREC_ROOT = "/kaggle/working/ColdRec"
COLDREC_COMMIT = "1108192"
DATASETS = ("MOOCCubeX_raw", "MOOCCubeX_fused", "MOOCCubeX_llm")
ID_MARK = "[patch-id]"
COLD_MARK = "[patch-cold-valid]"


def _read(p):
    with open(p, "r", encoding="utf-8") as f:
        return f.read()


def _write(p, s):
    with open(p, "w", encoding="utf-8") as f:
        f.write(s)


def patch_coldrec_source(root=COLDREC_ROOT, restore=True):
    """Vá model/__init__.py, util/databuilder.py, model/USIM.py. Khôi phục bản gốc từ git trước
    (xóa mọi patch/hook cũ, kể cả hook AUTO_EXPORT trong main.py của các notebook trước)."""
    if restore:
        subprocess.run(
            ["git", "-C", root, "checkout", "--", "main.py", "model/__init__.py",
             "util/databuilder.py", "model/USIM.py"],
            check=True,
        )
    head = subprocess.run(["git", "-C", root, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    assert head.startswith(COLDREC_COMMIT), f"ColdRec phải ở commit {COLDREC_COMMIT}, hiện là {head}"

    # (1) model/__init__.py: commit này import SimCSR nhưng không có tệp SimCSR.py
    p = os.path.join(root, "model", "__init__.py")
    s = _read(p)
    assert not os.path.exists(os.path.join(root, "model", "SimCSR.py")), "SimCSR.py tồn tại: không cần vá (1)"
    s = re.sub(r"^from \.SimCSR import SimCSR[ \t]*\n", "", s, flags=re.M)      # dòng import
    s = re.sub(r"^[ \t]*'SimCSR':[ \t]*SimCSR,[ \t]*\n", "", s, flags=re.M)    # mục trong AVAILABLE_MODELS
    assert "SimCSR" not in s, "model/__init__.py vẫn còn SimCSR"
    _write(p, s)
    compile(s, p, "exec")

    # (2) util/databuilder.py: mã nội bộ == mã gốc (ColdStartDataBuilder, lớp đầu tiên của tệp).
    #     Mặc định ColdRec đánh lại mã theo thứ tự xuất hiện trong csv => backbone bị gắn nhầm, và
    #     khóa không xuất hiện trong csv (340/753 khóa lạnh) không có mã nội bộ.
    p = os.path.join(root, "util", "databuilder.py")
    s = _read(p)
    assert ID_MARK not in s
    anchor = "    def generate_set(self):\n"
    pos = s.index(anchor)                                   # lần xuất hiện đầu tiên
    assert pos < s.index("class ColdStartMetaDataBuilder"), "anchor không nằm trong ColdStartDataBuilder"
    block = (
        f"        # {ID_MARK} mã nội bộ == mã gốc; gán đúng nội dung khóa/học viên theo mã gốc\n"
        "        for u in range(self.user_num):\n"
        "            self.user[u] = u\n"
        "            self.id2user[u] = u\n"
        "            if self.source_user_content is not None and u < self.source_user_content.shape[0]:\n"
        "                self.mapped_user_content[u] = self.source_user_content[u]\n"
        "        for i in range(self.item_num):\n"
        "            self.item[i] = i\n"
        "            self.id2item[i] = i\n"
        "            if self.source_item_content is not None and i < self.source_item_content.shape[0]:\n"
        "                self.mapped_item_content[i] = self.source_item_content[i]\n"
    )
    s = s[: pos + len(anchor)] + block + s[pos + len(anchor):]
    _write(p, s)
    compile(s, p, "exec")

    # (3) model/USIM.py: dừng sớm/chọn checkpoint theo khóa lạnh thay vì 'all'
    p = os.path.join(root, "model", "USIM.py")
    s = _read(p)
    n_all = s.count("valid_type='all')")
    assert n_all == 1, f"USIM.py: kỳ vọng đúng 1 chỗ valid_type='all', tìm thấy {n_all}"
    s = s.replace("valid_type='all')", f"valid_type='cold')  # {COLD_MARK}", 1)
    _write(p, s)
    compile(s, p, "exec")

    print("[OK] (1) __init__.py không còn SimCSR | (2) databuilder: ID nội bộ == ID gốc | (3) USIM: valid_type='cold'")


def write_backbone_and_info_dict(ad, datasets=DATASETS, root=COLDREC_ROOT, data_root="/kaggle/working/ColdRec_Data",
                                 expect_cold=753, expect_warm=2401):
    """(4) Lưu backbone LightGCN chung đúng tên ColdRec đọc khi --backbone LightGCN.
       (5) Đặt lại info_dict.pkl: đủ 753 khóa lạnh + 2.401 khóa ấm, giữ nguyên index gốc."""
    emb_dir = os.path.join(root, "emb")
    os.makedirs(emb_dir, exist_ok=True)
    g_u = np.ascontiguousarray(ad.g_u, dtype=np.float32)
    g_i = np.ascontiguousarray(ad.g_i, dtype=np.float32)
    cold = np.asarray(ad.cold_idx, dtype=np.int32)
    warm = np.asarray(ad.warm_active_idx, dtype=np.int32)
    assert len(cold) == expect_cold and len(warm) == expect_warm, (len(cold), len(warm))
    assert not (set(cold.tolist()) & set(warm.tolist()))
    assert np.allclose(g_i[cold], 0.0), "g_i của khóa lạnh phải bằng 0"

    for ds in datasets:
        for side, arr in (("user", g_u), ("item", g_i)):
            out = os.path.join(emb_dir, f"{ds}_cold_item_LightGCN_{side}_emb.pt")
            if os.path.lexists(out):
                os.remove(out)                       # bỏ symlink cũ của notebook trước (tránh ghi xuyên symlink)
            torch.save(torch.from_numpy(arr.copy()), out)
            back = torch.load(out, map_location="cpu")
            assert tuple(back.shape) == arr.shape and np.array_equal(back.numpy(), arr), out
        print(f"[OK] backbone: emb/{ds}_cold_item_LightGCN_{{user,item}}_emb.pt  "
              f"user{tuple(g_u.shape)} item{tuple(g_i.shape)}")

        f = os.path.join(root, "data", ds, "cold_item", "info_dict.pkl")
        if not os.path.exists(f):
            f = os.path.join(data_root, ds, "cold_item", "info_dict.pkl")
        with open(f, "rb") as fh:
            info = pickle.load(fh)
        info["cold_item"] = cold.copy()
        info["warm_item"] = warm.copy()
        with open(f, "wb") as fh:
            pickle.dump(info, fh, protocol=4)
        with open(f, "rb") as fh:
            chk = pickle.load(fh)
        assert len(chk["cold_item"]) == expect_cold and len(chk["warm_item"]) == expect_warm
        assert np.array_equal(chk["cold_item"], cold) and np.array_equal(chk["warm_item"], warm)
        assert chk["user_num"] == g_u.shape[0] and chk["item_num"] == g_i.shape[0]
        print(f"[OK] info_dict.pkl [{ds}]: cold_item={len(chk['cold_item'])}, warm_item={len(chk['warm_item'])}")


_VERIFY_SCRIPT = r"""
import pickle, sys, types, numpy as np
sys.path.insert(0, '.')
from util.loader import DataLoader
from util.databuilder import ColdStartDataBuilder
import model  # kiểm tra model/__init__.py import được (đã bỏ SimCSR)
ds = sys.argv[1]
L = lambda n: DataLoader.load_data_set(f'./data/{ds}/cold_item/{n}.csv')
info = pickle.load(open(f'./data/{ds}/cold_item/info_dict.pkl', 'rb'))
content = np.load(f'./data/{ds}/{ds}_item_content.npy')
b = ColdStartDataBuilder(L('warm_train'), L('warm_val'), L('cold_item_val'), L('overall_val'),
                         L('warm_test'), L('cold_item_test'), L('overall_test'),
                         info['user_num'], info['item_num'], info['warm_user'], info['warm_item'],
                         info['cold_user'], info['cold_item'], None, content)
assert all(b.user[u] == u for u in range(info['user_num'])) and len(b.user) == info['user_num']
assert all(b.item[i] == i for i in range(info['item_num'])) and len(b.item) == info['item_num']
assert all(b.id2user[u] == u for u in range(info['user_num'])) and all(b.id2item[i] == i for i in range(info['item_num']))
assert np.array_equal(b.mapped_cold_item_idx, np.asarray(info['cold_item'])), 'khóa lạnh bị gắn nhầm mã'
assert np.array_equal(b.mapped_warm_item_idx, np.asarray(info['warm_item'])), 'khóa ấm bị gắn nhầm mã'
n = info['item_num']
assert np.allclose(b.mapped_item_content[:n], content[:n]), 'nội dung khóa không khớp mã gốc'
print('VERIFY_OK', ds, 'cold', len(b.mapped_cold_item_idx), 'warm', len(b.mapped_warm_item_idx))
"""


def verify_identity_mapping(ds, root=COLDREC_ROOT, python=sys.executable):
    """Dựng ColdStartDataBuilder y như main.py và kiểm tra mã nội bộ == mã gốc (chạy trong tiến trình riêng)."""
    r = subprocess.run([python, "-c", _VERIFY_SCRIPT, ds], cwd=root, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip().splitlines()
    print("\n".join(out[-3:]))
    assert r.returncode == 0 and any(l.startswith("VERIFY_OK") for l in out), f"verify_identity_mapping({ds}) thất bại"

Writing coldrec_fix.py


In [7]:
# Áp dụng coldrec_fix.py MỘT lần (idempotent): vá mã nguồn + ghi backbone LightGCN + đặt lại info_dict.pkl.
import importlib
import subprocess
import sys

sys.path.insert(0, "/kaggle/working")
import coldrec_fix
importlib.reload(coldrec_fix)
from coldrec_adapter import ColdRecDataAdapter

ad = ColdRecDataAdapter()                      # chạy lại mọi assert của giao thức (753 khóa lạnh, 2.401 khóa ấm, MD5, ...)

coldrec_fix.patch_coldrec_source()             # (1) bỏ SimCSR  (2) ID nội bộ == ID gốc  (3) USIM valid_type='cold'
coldrec_fix.write_backbone_and_info_dict(ad)   # (4) emb/{ds}_cold_item_LightGCN_{user,item}_emb.pt  (5) info_dict.pkl
for _ds in coldrec_fix.DATASETS:
    coldrec_fix.verify_identity_mapping(_ds)   # dựng ColdStartDataBuilder như main.py và kiểm tra mã nội bộ == mã gốc

print(subprocess.run(["git", "-C", coldrec_fix.COLDREC_ROOT, "diff", "--stat"], capture_output=True, text=True).stdout)

=== KIỂM TRA MD5 VÀ BẢNG ÁNH XẠ CHỈ SỐ CHUNG ===
MD5 g_u.npy: 7a170558642de73d301970e8ce7f0a86 (Kỳ vọng: 7a170558...)
MD5 g_i.npy: d17ad9dfc4f4d6cb6de915c8649070d7 (Kỳ vọng: d17ad9df...)
MD5 raw_emb: f80445649ef8a8217d048bd3e7ac0ad5 | MD5 llm_emb: fc1819307be227a1544e1be0a4017393
MD5 llm_3d_emb: 1af436095463ad0af3ef1203ec0ebd6c
=== HOÀN TẤT KIỂM TRA: MỌI ASSERT ĐỀU ĐẠT CHUẨN! ===

[OK] (1) __init__.py không còn SimCSR | (2) databuilder: ID nội bộ == ID gốc | (3) USIM: valid_type='cold'
[OK] backbone: emb/MOOCCubeX_raw_cold_item_LightGCN_{user,item}_emb.pt  user(98220, 64) item(3769, 64)
[OK] info_dict.pkl [MOOCCubeX_raw]: cold_item=753, warm_item=2401
[OK] backbone: emb/MOOCCubeX_fused_cold_item_LightGCN_{user,item}_emb.pt  user(98220, 64) item(3769, 64)
[OK] info_dict.pkl [MOOCCubeX_fused]: cold_item=753, warm_item=2401
[OK] backbone: emb/MOOCCubeX_llm_cold_item_LightGCN_{user,item}_emb.pt  user(98220, 64) item(3769, 64)
[OK] info_dict.pkl [MOOCCubeX_llm]: cold_item=753, warm_item=240

In [8]:
# Hàm hỗ trợ: chạy USIM bằng main.py, kiểm tra log, nạp vector nhúng, chấm điểm bằng evaluator của nhóm.
import itertools
import json
import os
import random
import re
import subprocess
import sys
import time

import numpy as np
import torch
from coldrec_adapter import ColdRecDataAdapter, evaluate, evaluate_coldrec_embeddings

COLDREC_ROOT = "/kaggle/working/ColdRec"
EMB_DIR = f"{COLDREC_ROOT}/emb"
RESULT_DIR = "/kaggle/working/results_usim"
LOG_DIR = f"{RESULT_DIR}/logs"
PER_SEED_DIR = f"{RESULT_DIR}/per_seed"
SCORE_DIR = "/kaggle/working/scores/USIM"          # tệp điểm theo học viên (đúng quy ước scores/<Mô hình>/)
for _d in (RESULT_DIR, LOG_DIR, PER_SEED_DIR, SCORE_DIR):
    os.makedirs(_d, exist_ok=True)

SEEDS = [42, 2023, 2024, 1007, 999]
TUNE_SEED = 42
EPOCHS, EARLY_STOP = 200, 10                        # --epochs 200 --early_stop 10
USIM_BS = 64                                        # xem ghi chú ở cuối notebook: bs mặc định 4096 gây OOM (one_hot B×k×98.221)
LIGHTGCN_ID_NDCG20 = 0.0165                         # mốc LightGCN thuần ID trên tập (a)
SLOW_RUN_SECONDS = 30 * 60                          # lượt chạy mặc định > 30 phút thì bỏ USIM

DEFAULT_CFG = dict(actor_lr=5e-4, critic_lr=1e-3, usim_k=10, usim_reward_weight=0.5)
SPACE = dict(
    actor_lr=[1e-4, 2.5e-4, 5e-4, 1e-3],
    critic_lr=[1e-4, 5e-4, 1e-3, 2e-3],
    usim_k=[5, 10, 20],
    usim_reward_weight=[0.25, 0.5, 0.75],
)
REQ_BACKBONE = "USIM: loaded backbone embeddings (LightGCN)"
REQ_COLD = "Evaluating the model under the cold setting"


def build_trials():
    """20 cấu hình bằng random.Random(42) từ lưới con 144 tổ hợp (giống cách làm của ALDI: mặc định luôn có mặt)."""
    grid = [dict(zip(SPACE, v)) for v in itertools.product(*SPACE.values())]
    assert len(grid) == 144 and DEFAULT_CFG in grid
    trials = random.Random(42).sample(grid, 20)
    if DEFAULT_CFG not in trials:
        trials[-1] = dict(DEFAULT_CFG)               # vẫn đúng 20 cấu hình
    return trials


TRIALS = build_trials()


# nạp vector nhúng + chấm điểm
def usim_emb_paths(ds):
    return (f"{EMB_DIR}/{ds}_cold_item_USIM_user_emb.pt", f"{EMB_DIR}/{ds}_cold_item_USIM_item_emb.pt")


def load_usim_embeddings(adapter, ds):
    """Đọc emb/{ds}_cold_item_USIM_{user,item}_emb.pt và kiểm tra tính hợp lệ trước khi chấm điểm."""
    up, ip = usim_emb_paths(ds)
    for p in (up, ip):
        assert os.path.isfile(p), f"Thiếu {p}: lần chạy không lưu được vector nhúng"
    u = torch.load(up, map_location="cpu").float().numpy()
    i = torch.load(ip, map_location="cpu").float().numpy()
    assert u.shape == (98220, 64) and i.shape == (3769, 64), (u.shape, i.shape)
    assert np.isfinite(u).all() and np.isfinite(i).all(), "Vector nhúng có NaN/Inf"
    zero = int((np.abs(i[adapter.cold_idx]).sum(1) == 0).sum())
    assert zero == 0, f"{zero} khóa lạnh mang vector 0: kiểm tra bước (2), (5) của coldrec_fix.py"
    # Backbone phải được gắn đúng mã: USIM trả nguyên g_u cho học viên và g_i cho khóa ấm có tương tác
    assert np.allclose(u, adapter.g_u, atol=1e-6), "Vector học viên khác g_u: backbone bị gắn nhầm mã"
    w = adapter.warm_active_idx
    assert np.allclose(i[w], adapter.g_i[w], atol=1e-6), "Vector khóa ấm khác g_i: mã khóa bị lệch"
    return u, i


@torch.no_grad()
def evaluate_coldrec_val_only(adapter, u_emb, i_emb, model_name, variant_name="raw", seed=42,
                              device="cuda:0" if torch.cuda.is_available() else "cpu",
                              save_reps=True, verbose=True):
    """NDCG@20 / Recall@20 trên COLD-VAL (251 khóa, tập ứng viên (a)); che lịch sử train; KHÔNG xem tập test.
    Đây là tiêu chí chọn cấu hình của giao thức (không dùng số ColdRec tự in)."""
    u_t = torch.as_tensor(u_emb, dtype=torch.float32, device=device)
    i_t = torch.as_tensor(i_emb, dtype=torch.float32, device=device)
    assert u_t.shape == (98220, 64) and i_t.shape == (3769, 64), (u_t.shape, i_t.shape)
    assert not torch.isnan(u_t).any() and not torch.isnan(i_t).any(), "Có giá trị NaN trong embedding!"
    score_fn = lambda ub: (u_t[ub.to(device)] @ i_t.t()).cpu()
    res_val = evaluate(score_fn, adapter.val_target_dict, cand=adapter.cand_val_a, mask=adapter.hist_tr,
                       ks=(5, 10, 20), tie_seed=seed)
    v_ndcg, v_rec = float(np.mean(res_val["NDCG@20"])), float(np.mean(res_val["Recall@20"]))
    if save_reps:
        rep_dir = os.path.join(adapter.output_dir, f"results_{model_name}", "default_val_reps")
        os.makedirs(rep_dir, exist_ok=True)
        np.save(os.path.join(rep_dir, f"u_e_{variant_name}_seed{seed}.npy"), u_t.cpu().numpy())
        np.save(os.path.join(rep_dir, f"v_e_{variant_name}_seed{seed}.npy"), i_t.cpu().numpy())
    if verbose:
        print(f"[{model_name.upper()} (seed={seed}, {variant_name})] Cold-Val NDCG@20 = {v_ndcg:.4f} | "
              f"Cold-Val Recall@20 = {v_rec:.4f}")
    return v_ndcg, v_rec


# phân tích log của ColdRec
_EP = re.compile(r"^Epoch:\s*(\d+),.*NDCG:\s*(\S+)\s*$")


def parse_log(text):
    cur, best, mode = [], None, None
    for line in text.splitlines():
        s = line.strip()
        if s == "*Current Performance*":
            mode = "cur"
            continue
        if s.startswith("*Best") and s.endswith("Performance*"):
            mode = "best"
            continue
        if mode:
            m = _EP.match(s)
            if m:
                rec = (int(m.group(1)), float(m.group(2)))
                if mode == "cur":
                    cur.append(rec)
                else:
                    best = rec
            mode = None
    stop = re.findall(r"Stopping early at epoch (\d+)", text)
    return dict(
        backbone_loaded=REQ_BACKBONE in text,
        backbone_missing="USIM: backbone .pt not found" in text,
        cold_eval=REQ_COLD in text,
        wrong_valid=bool(re.search(r"Evaluating the model under the (all|warm) setting", text)),
        stop_epoch=int(stop[-1]) if stop else None,
        n_eval=len(cur),
        cur=cur,
        best_epoch=best[0] if best else None,
        best_ndcg=best[1] if best else None,
    )


def check_log(info, early_stop=EARLY_STOP):
    """Trả về (lỗi_nghiêm_trọng, bị_đứng_yên). Điều kiện (i) và (ii) của Review Y4"""
    fatal = []
    if not info["backbone_loaded"]:
        fatal.append(f"Log thiếu dòng '{REQ_BACKBONE}'" + (" (ColdRec báo 'backbone .pt not found': dùng Xavier ngẫu nhiên)"
                                                           if info["backbone_missing"] else ""))
    if not info["cold_eval"]:
        fatal.append(f"Log thiếu dòng '{REQ_COLD}'")
    if info["wrong_valid"]:
        fatal.append("Log có kiểm định theo tập 'all'/'warm': patch (3) valid_type='cold' chưa có hiệu lực")
    vals = [v for _, v in info["cur"]]
    stalled = bool(
        (info["stop_epoch"] is not None and info["stop_epoch"] <= early_stop + 1 and info["best_epoch"] == 1)
        or (len(vals) >= 2 and max(vals) - min(vals) < 1e-12)
    )
    return fatal, stalled


_ECHO = re.compile(r"^(USIM:|Stopping early|Traceback|\w*Error)")


def run_usim(inp, seed, cfg=None, tag="run", epochs=EPOCHS, early_stop=EARLY_STOP, bs=USIM_BS, echo=True):
    """Chạy `python main.py --model USIM ...` (cwd = ColdRec), ghi log, phân tích log. Ném lỗi nếu main.py thoát bất thường."""
    ds = f"MOOCCubeX_{inp}"
    cfg = {**DEFAULT_CFG, **(cfg or {})}
    for p in usim_emb_paths(ds):                      # tránh đọc nhầm vector của lần chạy trước khi lần này hỏng
        if os.path.lexists(p):
            os.remove(p)
    cmd = [sys.executable, "main.py", "--dataset", ds, "--model", "USIM", "--cold_object", "item",
           "--backbone", "LightGCN", "--emb_size", "64", "--epochs", str(epochs), "--early_stop", str(early_stop),
           "--bs", str(bs), "--seed", str(seed), "--save_emb", "True",
           "--actor_lr", str(cfg["actor_lr"]), "--critic_lr", str(cfg["critic_lr"]),
           "--usim_k", str(cfg["usim_k"]), "--usim_reward_weight", str(cfg["usim_reward_weight"]),
           "--result_file", f"{LOG_DIR}/{tag}_result.txt", "--result_overwrite"]
    log_path = f"{LOG_DIR}/{tag}.log"
    t0 = time.time()
    with open(log_path, "w", encoding="utf-8") as lf:
        proc = subprocess.Popen(cmd, cwd=COLDREC_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                                bufsize=1, env={**os.environ, "PYTHONUNBUFFERED": "1"})
        for line in proc.stdout:
            lf.write(line)
            if echo and _ECHO.search(line):
                print("    " + line.rstrip())
        rc = proc.wait()
    secs = time.time() - t0
    text = open(log_path, encoding="utf-8").read()
    if rc != 0:
        raise RuntimeError(f"main.py thoát mã {rc} ({tag}). 30 dòng cuối của {log_path}:\n" + "\n".join(text.splitlines()[-30:]))
    info = parse_log(text)
    fatal, stalled = check_log(info, early_stop)
    return dict(info=info, fatal=fatal, stalled=stalled, secs=secs, log=log_path, cfg=cfg)


def print_run_summary(r):
    i = r["info"]
    print(f"    log: {r['log']} | {r['secs'] / 60:.1f} phút | số lần kiểm định={i['n_eval']} | "
          f"dừng ở epoch={i['stop_epoch']} | checkpoint tốt nhất: epoch {i['best_epoch']} (NDCG nội bộ {i['best_ndcg']})")
    vals = [f"{v:.4f}" for _, v in i["cur"]]
    print("    NDCG cold-val nội bộ theo epoch:", " ".join(vals[:6]), "..." if len(vals) > 9 else "", " ".join(vals[-3:] if len(vals) > 6 else []))
    print(f"    (i) có '{REQ_BACKBONE}': {i['backbone_loaded']} | có '{REQ_COLD}': {i['cold_eval']} | (ii) bị đứng yên: {r['stalled']}")

Phải đạt đủ: (i) log có `USIM: loaded backbone embeddings (LightGCN)` và `Evaluating the model under the cold setting`; (ii) không dừng ở epoch 11 với NDCG đứng yên; (iii) NDCG@20 (a) > 0,0165. Không đạt thì **dừng**, không chạy tiếp.

In [9]:
# Chạy USIM ở cấu hình MẶC ĐỊNH (raw, seed 42): kiểm tra log + chấm điểm độc lập + lọc (sanity check) 
import json

if not torch.cuda.is_available():
    print("[CẢNH BÁO] Không có GPU: USIM trên CPU sẽ rất chậm. Bật accelerator GPU trong Kaggle.")


def default_run_and_sanity(inp="raw", seed=42):
    ds = f"MOOCCubeX_{inp}"
    print(f"=== USIM cấu hình mặc định {DEFAULT_CFG} | đầu vào={inp} | seed={seed} ===")
    r = run_usim(inp, seed, DEFAULT_CFG, tag=f"default_{inp}_seed{seed}")
    print_run_summary(r)
    for msg in r["fatal"]:
        print("    [LỖI LOG]", msg)

    # Chấm điểm ĐỘC LẬP: chỉ dùng vector nhúng USIM đã lưu, bỏ qua mọi số ColdRec tự in
    u, i = load_usim_embeddings(ad, ds)           # kiểm tra shape, NaN, không còn vector 0 ở 753 khóa lạnh, backbone khớp mã
    print(f"    vector nhúng hợp lệ: user{u.shape} item{i.shape}; 0/{len(ad.cold_idx)} khóa lạnh mang vector 0")
    val_ndcg, val_rec = evaluate_coldrec_val_only(ad, u, i, "usim", inp, seed)                 # cold-val, tập (a)
    res_a, res_b, res_g = evaluate_coldrec_embeddings(ad, u, i, "usim", inp, seed)             # cold-test, tập (a) và (b)
    ndcg_a = float(np.mean(res_a["NDCG@20"]))

    cond_i = not r["fatal"]
    cond_ii = not r["stalled"]
    cond_iii = ndcg_a > LIGHTGCN_ID_NDCG20
    passed = bool(cond_i and cond_ii and cond_iii)
    out = dict(input=inp, seed=seed, cfg=DEFAULT_CFG, passed=passed,
               cond_i_log=cond_i, cond_ii_not_stalled=cond_ii, cond_iii_beats_lightgcn_id=cond_iii,
               cold_val_ndcg20=val_ndcg, cold_val_recall20=val_rec,
               cold_test_a_ndcg20=ndcg_a, cold_test_a_recall20=float(np.mean(res_a["Recall@20"])),
               all_items_b_ndcg20=float(np.mean(res_b["NDCG@20"])),
               stop_epoch=r["info"]["stop_epoch"], best_epoch=r["info"]["best_epoch"],
               minutes=round(r["secs"] / 60, 1), log=r["log"])
    json.dump(out, open(f"{RESULT_DIR}/sanity_{inp}.json", "w"), indent=1, ensure_ascii=False)

    print("\n--- KẾT QUẢ LỌC ---")
    print(f"(i)   log có 2 dòng bắt buộc           : {'ĐẠT' if cond_i else 'KHÔNG ĐẠT'}")
    print(f"(ii)  không dừng ở epoch 11, NDCG đứng yên: {'ĐẠT' if cond_ii else 'KHÔNG ĐẠT'}")
    print(f"(iii) NDCG@20 (a) cold-test = {ndcg_a:.4f} so với LightGCN thuần ID {LIGHTGCN_ID_NDCG20}: "
          f"{'ĐẠT' if cond_iii else 'KHÔNG ĐẠT'}   (cold-val NDCG@20 = {val_ndcg:.4f})")
    if r["secs"] > SLOW_RUN_SECONDS:
        print(f"[LƯU Ý THỜI GIAN] Lượt mặc định mất {r['secs'] / 60:.1f} phút (> 30 phút): theo Mục 5.3 của báo cáo "
              f"không kịp tinh chỉnh 20 cấu hình × 3 đầu vào × 5 seed; cân nhắc bỏ USIM thay vì giảm số cấu hình.")
    if passed:
        print(">>> ĐẠT cả 3 điều kiện: được phép tinh chỉnh 20 cấu hình và chạy 5 seed.")
    else:
        print(">>> KHÔNG ĐẠT: DỪNG. Không tinh chỉnh, không chạy tiếp; ghi USIM vào phần hạn chế. ")
    return passed


SANITY_PASSED = default_run_and_sanity("raw", 42)

=== USIM cấu hình mặc định {'actor_lr': 0.0005, 'critic_lr': 0.001, 'usim_k': 10, 'usim_reward_weight': 0.5} | đầu vào=raw | seed=42 ===
    USIM: loaded backbone embeddings (LightGCN)
    Stopping early at epoch 18.
    log: /kaggle/working/results_usim/logs/default_raw_seed42.log | 15.9 phút | số lần kiểm định=18 | dừng ở epoch=18 | checkpoint tốt nhất: epoch 8 (NDCG nội bộ 0.00309)
    NDCG cold-val nội bộ theo epoch: 0.0029 0.0005 0.0004 0.0004 0.0007 0.0008 ... 0.0008 0.0008 0.0014
    (i) có 'USIM: loaded backbone embeddings (LightGCN)': True | có 'Evaluating the model under the cold setting': True | (ii) bị đứng yên: False
    vector nhúng hợp lệ: user(98220, 64) item(3769, 64); 0/753 khóa lạnh mang vector 0
[USIM (seed=42, raw)] Cold-Val NDCG@20 = 0.0180 | Cold-Val Recall@20 = 0.0505

--- KẾT QUẢ LỌC ---
(i)   log có 2 dòng bắt buộc           : ĐẠT
(ii)  không dừng ở epoch 11, NDCG đứng yên: ĐẠT
(iii) NDCG@20 (a) cold-test = 0.0053 so với LightGCN thuần ID 0.0165: KHÔNG ĐẠT   (

## Tinh chỉnh 20 cấu hình (seed 42, cold-val) rồi chạy 5 seed {42, 2023, 2024, 1007, 999}
Chạy `raw` và `fused` song song trên hai tài khoản Kaggle (đổi `INPUTS_TO_RUN`), sau đó `llm`. Kết quả ghi vào `scores/USIM/` và `results_usim/` (log 20 cấu hình: `trials_{inp}.json`).

In [10]:
# # Tinh chỉnh 20 cấu hình (seed 42, chọn theo NDCG@20 cold-val tập (a)) rồi chạy đủ 5 seed cho từng đầu vào.
# # Có thể chạy lại ô này sau khi phiên bị ngắt: các lượt đã xong được đọc lại từ results_usim/ (không chạy lại).
# import json

# # Chạy song song 2 tài khoản Kaggle: tài khoản A đặt ["raw"], tài khoản B đặt ["fused"]; sau đó đến ["llm"].
# INPUTS_TO_RUN = ["raw"]
# # Tài khoản chưa chạy ô 'cấu hình mặc định' (ô trước) mà đã được xác nhận đạt 3 điều kiện ở tài khoản khác: đặt True.
# SANITY_CONFIRMED_ELSEWHERE = False


# def require_sanity():
#     if SANITY_CONFIRMED_ELSEWHERE:
#         return
#     path = f"{RESULT_DIR}/sanity_raw.json"
#     assert os.path.isfile(path), "Chưa có kết quả lọc: chạy ô 'cấu hình mặc định' trước."
#     assert json.load(open(path))["passed"], "USIM KHÔNG đạt điều kiện lọc: dừng, không tinh chỉnh."


# def tune_usim(inp, seed=TUNE_SEED):
#     ds = f"MOOCCubeX_{inp}"
#     path = f"{RESULT_DIR}/trials_{inp}.json"
#     recs = json.load(open(path)) if os.path.isfile(path) else []
#     done = {r["trial"] for r in recs}
#     print(f"=== TINH CHỈNH USIM [{inp}] (seed={seed}): {len(TRIALS)} cấu hình; đã có {len(done)} ===")
#     for idx, cfg in enumerate(TRIALS, 1):
#         if idx in done:
#             continue
#         r = run_usim(inp, seed, cfg, tag=f"tune_{inp}_t{idx:02d}_seed{seed}", echo=False)
#         if r["fatal"]:
#             raise RuntimeError(f"Trial {idx} [{inp}] vi phạm điều kiện log: {r['fatal']} ({r['log']})")
#         u, i = load_usim_embeddings(ad, ds)
#         v_ndcg, v_rec = evaluate_coldrec_val_only(ad, u, i, "usim", inp, seed, save_reps=False, verbose=False)
#         rec = dict(trial=idx, val_ndcg20=v_ndcg, val_recall20=v_rec, best_ep=r["info"]["best_epoch"],
#                    stop_epoch=r["info"]["stop_epoch"], stalled=r["stalled"], minutes=round(r["secs"] / 60, 1), cfg=cfg)
#         recs.append(rec)
#         json.dump(sorted(recs, key=lambda x: x["trial"]), open(path, "w"), indent=1)   # lưu log 20 cấu hình 
#         print(f"  Trial {idx:02d}/{len(TRIALS)} | val_NDCG@20={v_ndcg:.4f} (ep={rec['best_ep']}, dừng={rec['stop_epoch']}"
#               f"{', ĐỨNG YÊN' if r['stalled'] else ''}, {rec['minutes']} phút) | {cfg}")
#     recs = sorted(recs, key=lambda x: x["trial"])
#     assert [r["trial"] for r in recs] == list(range(1, len(TRIALS) + 1))
#     best = max(recs, key=lambda x: x["val_ndcg20"])        # hòa điểm: lấy trial có số thứ tự nhỏ hơn
#     json.dump(best, open(f"{RESULT_DIR}/best_cfg_{inp}.json", "w"), indent=1)
#     print(f"-> Cấu hình tốt nhất [{inp}]: trial {best['trial']} | val_NDCG@20={best['val_ndcg20']:.4f} "
#           f"(ep={best['best_ep']}) | {best['cfg']}")
#     return best["cfg"], best


# def run_five_seeds(inp, cfg):
#     ds = f"MOOCCubeX_{inp}"
#     names = ("a", "trung", "moi")
#     per = []
#     print(f"=== CHẠY 5 SEED USIM [{inp}] với cấu hình {cfg} ===")
#     for s in SEEDS:
#         npz_p, js_p = f"{PER_SEED_DIR}/{inp}_seed{s}.npz", f"{PER_SEED_DIR}/{inp}_seed{s}.json"
#         if os.path.isfile(npz_p) and os.path.isfile(js_p):
#             z = np.load(npz_p)
#             rec = dict(arrays={k: z[k] for k in names}, **json.load(open(js_p)))
#             print(f"  [Seed {s}] đã có kết quả, bỏ qua chạy lại | (a) NDCG@20={rec['a_NDCG20']:.4f}")
#         else:
#             r = run_usim(inp, s, cfg, tag=f"final_{inp}_seed{s}")
#             print_run_summary(r)
#             if r["fatal"]:
#                 raise RuntimeError(f"Seed {s} [{inp}] vi phạm điều kiện log: {r['fatal']} ({r['log']})")
#             if r["stalled"]:
#                 print(f"    [CẢNH BÁO] Seed {s} [{inp}] dừng sớm với NDCG đứng yên; ghi vào báo cáo cho GVHD.")
#             u, i = load_usim_embeddings(ad, ds)
#             res_a, res_b, res_g = evaluate_coldrec_embeddings(ad, u, i, "usim", inp, s)
#             arrays = dict(a=np.asarray(res_a["NDCG@20"], dtype=np.float32),
#                           trung=np.asarray(res_g["trung"]["NDCG@20"], dtype=np.float32),
#                           moi=np.asarray(res_g["moi"]["NDCG@20"], dtype=np.float32))
#             assert len(arrays["a"]) == 3340 and len(arrays["trung"]) == 1028 and len(arrays["moi"]) == 2844
#             meta = dict(seed=s, stalled=bool(r["stalled"]), best_ep=r["info"]["best_epoch"], stop_epoch=r["info"]["stop_epoch"],
#                         minutes=round(r["secs"] / 60, 1),
#                         a_R20=float(np.mean(res_a["Recall@20"])), a_NDCG20=float(np.mean(res_a["NDCG@20"])),
#                         b_R20=float(np.mean(res_b["Recall@20"])), b_NDCG20=float(np.mean(res_b["NDCG@20"])))
#             for g in ("trung", "moi"):
#                 meta[f"{g}_R20"] = float(np.mean(res_g[g]["Recall@20"]))
#                 meta[f"{g}_NDCG20"] = float(np.mean(res_g[g]["NDCG@20"]))
#             np.savez(npz_p, **arrays)
#             json.dump(meta, open(js_p, "w"), indent=1)
#             rec = dict(arrays=arrays, **meta)
#             print(f"  [Seed {s}] (a) R@20={meta['a_R20']:.4f}, NDCG@20={meta['a_NDCG20']:.4f} | (b) NDCG@20={meta['b_NDCG20']:.4f} | "
#                   f"trùng NDCG@20={meta['trung_NDCG20']:.4f}, mới NDCG@20={meta['moi_NDCG20']:.4f}")
#         per.append(rec)

#     # Lưu NDCG@20 theo học viên (trung bình 5 seed) vào scores/USIM/, giống evaluate_5_seeds của ALDI
#     mean_a = np.mean(np.stack([p["arrays"]["a"] for p in per]), axis=0)
#     assert len(mean_a) == 3340
#     np.save(f"{SCORE_DIR}/user_ndcg20_usim_{inp}.npy", mean_a)
#     for g, n in (("trung", 1028), ("moi", 2844)):
#         mg = np.mean(np.stack([p["arrays"][g] for p in per]), axis=0)
#         assert len(mg) == n
#         np.save(f"{SCORE_DIR}/user_ndcg20_usim_{inp}_{g}.npy", mg)

#     col = lambda k: [p[k] for p in per]
#     sd = lambda k: float(np.std(col(k), ddof=1))
#     summary = dict(input=inp, cfg=cfg, seeds=SEEDS,
#                    a_R20_mean=float(np.mean(col("a_R20"))), a_R20_std=sd("a_R20"),
#                    a_NDCG20_mean=float(np.mean(col("a_NDCG20"))), a_NDCG20_std=sd("a_NDCG20"),
#                    b_R20_mean=float(np.mean(col("b_R20"))), b_NDCG20_mean=float(np.mean(col("b_NDCG20"))),
#                    trung_NDCG20_mean=float(np.mean(col("trung_NDCG20"))), moi_NDCG20_mean=float(np.mean(col("moi_NDCG20"))),
#                    trung_R20_mean=float(np.mean(col("trung_R20"))), moi_R20_mean=float(np.mean(col("moi_R20"))),
#                    stalled_seeds=[p["seed"] for p in per if p["stalled"]])
#     json.dump(summary, open(f"{RESULT_DIR}/usim_summary_{inp}.json", "w"), indent=2, ensure_ascii=False)
#     print(f"=> TỔNG KẾT USIM [{inp}]: Cold-test (a) NDCG@20 = {summary['a_NDCG20_mean']:.4f} ± {summary['a_NDCG20_std']:.4f} | "
#           f"R@20 = {summary['a_R20_mean']:.4f} ± {summary['a_R20_std']:.4f} | "
#           f"nhóm TRÙNG/MỚI NDCG@20 = {summary['trung_NDCG20_mean']:.4f} / {summary['moi_NDCG20_mean']:.4f}"
#           + (f" | seed đứng yên: {summary['stalled_seeds']}" if summary["stalled_seeds"] else ""))
#     return summary


# def tune_and_run_usim(inp):
#     require_sanity()
#     cfg, _ = tune_usim(inp)
#     return run_five_seeds(inp, cfg)


# for _inp in INPUTS_TO_RUN:
#     tune_and_run_usim(_inp)

### Kiểm tra tệp điểm `scores/USIM/`

In [11]:
# # Kiểm tra đủ tệp scores/USIM/ (3 đầu vào × {toàn bộ, _trung, _moi}) và in bảng tổng hợp để chạy final-stats.ipynb.
# import glob
# import shutil

# rows = []
# for _inp in ("raw", "fused", "llm"):
#     sp = f"{RESULT_DIR}/usim_summary_{_inp}.json"
#     if not os.path.isfile(sp):
#         print(f"[CHƯA CÓ] {_inp}: chưa chạy xong (xem ô tinh chỉnh)")
#         continue
#     for _suffix, _n in (("", 3340), ("_trung", 1028), ("_moi", 2844)):
#         f = f"{SCORE_DIR}/user_ndcg20_usim_{_inp}{_suffix}.npy"
#         assert os.path.isfile(f) and len(np.load(f)) == _n, f"{f}: thiếu hoặc sai số học viên (kỳ vọng {_n})"
#     s = json.load(open(sp))
#     rows.append((_inp, s["a_NDCG20_mean"], s["a_NDCG20_std"], s["a_R20_mean"], s["a_R20_std"],
#                  s["trung_NDCG20_mean"], s["moi_NDCG20_mean"], s["stalled_seeds"]))

# print(f"{'Đầu vào':8} {'NDCG@20 (a)':>18} {'Recall@20 (a)':>18} {'trùng':>8} {'mới':>8}  seed đứng yên")
# for r in rows:
#     print(f"{r[0]:8} {r[1]:.4f} ± {r[2]:.4f}   {r[3]:.4f} ± {r[4]:.4f}   {r[5]:.4f}   {r[6]:.4f}  {r[7] or '-'}")
# print("\nTệp trong scores/USIM/:", *sorted(os.path.basename(p) for p in glob.glob(f"{SCORE_DIR}/*.npy")), sep="\n  ")
# if len(rows) == 3:
#     print("\nĐủ cả 3 đầu vào: sao chép scores/USIM/ vào repo rồi chạy final-stats.ipynb "
#           "(USIM tự vào các họ kiểm định E-RQ1 khi đủ tệp).")
#     shutil.make_archive("/kaggle/working/scores_USIM", "zip", SCORE_DIR)
#     print("Đã nén: /kaggle/working/scores_USIM.zip ; log 20 cấu hình: results_usim/trials_{raw,fused,llm}.json")
# else:
#     print("\nCHƯA đủ 3 đầu vào: USIM chưa được vào các họ kiểm định E-RQ1.")

---
*Ô ALDI bên dưới giữ nguyên (đang comment, không chạy).*

In [12]:
# %%writefile run_aldi.py
# import copy
# import itertools
# import json
# import os
# import random
# import time
# import numpy as np
# import torch
# import torch.nn as nn
# import torch.nn.functional as F

# from coldrec_adapter import ColdRecDataAdapter, evaluate, eval_subgroups

# DEV = "cuda:0" if torch.cuda.is_available() else "cpu"
# SEEDS = [42, 2023, 2024, 1007, 999]
# ROBUST_VAL_SEEDS = [42, 2023, 2024]


# # Kiến trúc ALDI chuyển nguyên bản từ mã TensorFlow chính thức 
# class ALDI(nn.Module):
#     def __init__(self, d_content, d_emb=64, h=200):
#         super().__init__()
#         self.f_I = nn.Sequential(
#             nn.Linear(d_content, h),
#             nn.BatchNorm1d(h),
#             nn.Tanh(),
#             nn.Linear(h, d_emb),
#         )
#         self.f_U = nn.Sequential(
#             nn.Linear(d_emb, h),
#             nn.BatchNorm1d(h),
#             nn.Tanh(),
#             nn.Linear(h, d_emb),
#         )
#         for m in self.modules():
#             if isinstance(m, nn.Linear):  # truncated_normal(stddev=0.01)
#                 nn.init.trunc_normal_(m.weight, std=0.01, a=-0.02, b=0.02)
#                 nn.init.zeros_(m.bias)

#     def l2(self):  # l2_regularizer: sum(w^2)/2, cả weight và bias
#         return (
#             sum(
#                 (m.weight ** 2).sum() + (m.bias ** 2).sum()
#                 for m in self.modules()
#                 if isinstance(m, nn.Linear)
#             )
#             / 2
#         )


# def aldi_loss(model, u_t, c_pos, c_neg, e_pos, e_neg, w_pos, cfg):
#     s_pos, s_neg = model.f_I(torch.cat([c_pos, c_neg])).chunk(2)  # BN trên lô dương + âm
#     u_s = model.f_U(u_t)
#     sp, sn = (u_s * s_pos).sum(-1), (u_s * s_neg).sum(-1)         # học sinh
#     tp, tn = (u_t * e_pos).sum(-1), (u_t * e_neg).sum(-1)         # giáo viên (LightGCN)
#     bce = F.binary_cross_entropy_with_logits
#     sup = bce(sp - sn, torch.ones_like(sp))
#     rank = (w_pos * bce(sp - sn, torch.sigmoid(tp - tn), reduction="none")).mean()
#     s_id = (s_pos * s_pos).sum(-1) - (s_pos @ s_neg.t()).mean(1)
#     t_id = (e_pos * e_pos).sum(-1) - (e_pos @ e_neg.t()).mean(1)
#     iden = (w_pos * bce(s_id, torch.sigmoid(t_id), reduction="none")).mean()
#     dist = ((tp - sp).abs() + (tn - sn).abs()).mean()
#     return (
#         sup
#         + cfg["alpha"] * rank
#         + cfg["beta"] * iden
#         + cfg["gamma"] * dist
#         + cfg["reg"] * model.l2()
#     )


# # Khởi tạo Adapter, trọng số tần suất khóa (tws) và hàm lấy mẫu 
# ad = ColdRecDataAdapter()
# g_u_t = torch.tensor(ad.g_u, dtype=torch.float32, device=DEV)
# g_i_t = torch.tensor(ad.g_i, dtype=torch.float32, device=DEV)

# u_tr = ad.tr_cold.user_id.astype(str).map(ad.user2idx).to_numpy()
# i_tr = ad.tr_cold.course_id.astype(str).map(ad.item2idx).to_numpy()
# deg_u = np.bincount(u_tr, minlength=ad.n_users)
# f = np.zeros(ad.n_items)
# np.add.at(f, i_tr, 1.0 / deg_u[u_tr])
# freq = np.ones(ad.n_items, dtype=np.float32)
# freq[ad.warm_active_idx] = f[ad.warm_active_idx]
# M = 4.0
# a = M / (ad.n_users / ad.n_items)
# w_item = torch.tensor(
#     np.clip(np.tanh(a * freq), 0, np.tanh(M)), dtype=torch.float32, device=DEV
# )

# warm_users = np.array(sorted(ad.hist_tr))


# def sample_epoch(rng):  # như utils.bpr_neg_samp
#     users = rng.choice(warm_users, size=len(u_tr), replace=True)
#     pos = np.array([ad.hist_tr[u][rng.integers(len(ad.hist_tr[u]))] for u in users])
#     neg = rng.choice(ad.warm_active_idx, size=len(u_tr), replace=True)
#     return users, pos, neg


# # 1.368 khóa không có tương tác trong train (753 khóa lạnh + 615 khóa ấm không hoạt động)
# gen_cols = torch.tensor(
#     np.setdiff1d(np.arange(ad.n_items), ad.warm_active_idx), device=DEV
# )


# def aldi_score_fn(model, content_t):
#     model.eval()
#     with torch.no_grad():
#         u_map, gen = model.f_U(g_u_t), model.f_I(content_t)

#     def fn(ub):
#         ub = ub.to(DEV)
#         sc = g_u_t[ub] @ g_i_t.t()                     # khóa ấm: điểm giáo viên
#         sc[:, gen_cols] = u_map[ub] @ gen[gen_cols].t() # khóa còn lại: học sinh
#         return sc

#     return fn


# def train_aldi(content_t, cfg, seed, max_epochs=200, patience=10, bs=1024):
#     random.seed(seed)
#     np.random.seed(seed)
#     torch.manual_seed(seed)
#     torch.cuda.manual_seed_all(seed)
#     rng = np.random.default_rng(seed)
#     model = ALDI(content_t.shape[1]).to(DEV)
#     opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])  # L2 nằm trong loss
#     w_t = w_item if cfg["tws"] else torch.ones_like(w_item)
#     best, best_state, bad, best_ep = -1.0, None, 0, 0
#     for ep in range(1, max_epochs + 1):
#         model.train()
#         users, pos, neg = (torch.tensor(x, device=DEV) for x in sample_epoch(rng))
#         for s in range(0, len(users) - bs, bs):  # như main.py: bỏ lô cuối thiếu
#             u, p, n = users[s : s + bs], pos[s : s + bs], neg[s : s + bs]
#             loss = aldi_loss(
#                 model,
#                 g_u_t[u],
#                 content_t[p],
#                 content_t[n],
#                 g_i_t[p],
#                 g_i_t[n],
#                 w_t[p],
#                 cfg,
#             )
#             opt.zero_grad()
#             loss.backward()
#             opt.step()
#         r = evaluate(
#             aldi_score_fn(model, content_t),
#             ad.val_target_dict,
#             mask=ad.hist_tr,
#             cand=ad.cand_val_a,
#             ks=(20,),
#             tie_seed=seed,
#         )
#         v = float(np.mean(r["NDCG@20"]))
#         if v > best:
#             best, best_state, bad, best_ep = (
#                 v,
#                 copy.deepcopy(model.state_dict()),
#                 0,
#                 ep,
#             )
#         else:
#             bad += 1
#             if bad > patience:
#                 break  # như main.py
#     model.load_state_dict(best_state)
#     return model, best, best_ep


# # Không gian siêu tham số 20 cấu hình 
# space = dict(
#     lr=[1e-4, 5e-4, 1e-3, 5e-3],
#     alpha=[0.1, 0.5, 0.9],
#     beta=[0.01, 0.05, 0.1],
#     gamma=[0.05, 0.1, 0.5],
#     reg=[1e-5, 1e-4, 1e-3],
#     tws=[0, 1],
# )
# grid = [dict(zip(space, v)) for v in itertools.product(*space.values())]
# trials = random.Random(42).sample(grid, 20)
# DEFAULT = dict(lr=1e-3, alpha=0.9, beta=0.05, gamma=0.1, reg=1e-4, tws=0)
# if DEFAULT not in trials:
#     trials[-1] = DEFAULT  # vẫn đúng 20 cấu hình


# def evaluate_5_seeds(inp_name, cfg, tag_suffix=""):
#     content_t = torch.tensor(ad.contents[inp_name], dtype=torch.float32, device=DEV)
#     out_dir = os.path.join(ad.output_dir, "results_aldi")
#     rep_dir = os.path.join(out_dir, "item_reps")
#     score_dir = os.path.join(ad.output_dir, "scores")
#     os.makedirs(rep_dir, exist_ok=True)
#     os.makedirs(score_dir, exist_ok=True)

#     metrics_a = {"Recall@20": [], "NDCG@20": []}
#     metrics_b = {"Recall@20": [], "NDCG@20": []}
#     sub_metrics = {
#         "trung": {"Recall@20": [], "NDCG@20": []},
#         "moi": {"Recall@20": [], "NDCG@20": []},
#     }
#     user_ndcg_a_seeds = []
#     user_ndcg_sub_seeds = {"trung": [], "moi": []}

#     n_params = sum(p.numel() for p in ALDI(content_t.shape[1]).parameters() if p.requires_grad)

#     for s in SEEDS:
#         t0 = time.time()
#         model, val_ndcg, best_ep = train_aldi(content_t, cfg, seed=s)
#         fn = aldi_score_fn(model, content_t)

#         kw = dict(mask=ad.hist_trva, ks=(5, 10, 20), tie_seed=s)
#         res_a = evaluate(fn, ad.test_target_dict, cand=ad.cand_test_a, **kw)
#         res_b = evaluate(fn, ad.test_target_dict, cand=ad.cand_test_b, **kw)
#         res_prq3 = eval_subgroups(
#             fn,
#             ad.test_target_dict,
#             ad.hist_trva,
#             ad.cand_test_a,
#             ad.groups_prq3,
#             ks=(5, 10, 20),
#             tie_seed=s,
#         )

#         # Lưu vector biểu diễn theo từng seed (bọc trong torch.no_grad)
#         model.eval()
#         with torch.no_grad():
#             u_map, gen = model.f_U(g_u_t), model.f_I(content_t)
#         save_key = f"{inp_name}{tag_suffix}"
#         np.save(os.path.join(rep_dir, f"u_e_{save_key}_seed{s}.npy"), u_map.detach().cpu().numpy())
#         np.save(os.path.join(rep_dir, f"v_e_{save_key}_seed{s}.npy"), gen.detach().cpu().numpy())

#         # Ghi nhận chỉ số
#         metrics_a["Recall@20"].append(float(np.mean(res_a["Recall@20"])))
#         metrics_a["NDCG@20"].append(float(np.mean(res_a["NDCG@20"])))
#         metrics_b["Recall@20"].append(float(np.mean(res_b["Recall@20"])))
#         metrics_b["NDCG@20"].append(float(np.mean(res_b["NDCG@20"])))

#         user_ndcg_a_seeds.append(np.asarray(res_a["NDCG@20"], dtype=np.float32))
#         for g in ("trung", "moi"):
#             sub_metrics[g]["Recall@20"].append(float(np.mean(res_prq3[g]["Recall@20"])))
#             sub_metrics[g]["NDCG@20"].append(float(np.mean(res_prq3[g]["NDCG@20"])))
#             user_ndcg_sub_seeds[g].append(np.asarray(res_prq3[g]["NDCG@20"], dtype=np.float32))

#         print(
#             f"  [Seed {s}] ep={best_ep:02d} | val_NDCG@20={val_ndcg:.4f} | "
#             f"(a) R@20={metrics_a['Recall@20'][-1]:.4f}, NDCG@20={metrics_a['NDCG@20'][-1]:.4f} | "
#             f"(b) NDCG@20={metrics_b['NDCG@20'][-1]:.4f} | "
#             f"trùng NDCG@20={sub_metrics['trung']['NDCG@20'][-1]:.4f}, "
#             f"mới NDCG@20={sub_metrics['moi']['NDCG@20'][-1]:.4f} ({time.time()-t0:.1f}s)"
#         )

#     # Lưu tệp điểm từng học viên (trung bình qua 5 seed)
#     save_key = f"{inp_name}{tag_suffix}"
#     mean_user_a = np.mean(np.stack(user_ndcg_a_seeds, axis=0), axis=0)
#     assert len(mean_user_a) == 3340
#     np.save(os.path.join(score_dir, f"user_ndcg20_aldi_{save_key}.npy"), mean_user_a)

#     for g, expected_len in (("trung", 1028), ("moi", 2844)):
#         mean_user_g = np.mean(np.stack(user_ndcg_sub_seeds[g], axis=0), axis=0)
#         assert len(mean_user_g) == expected_len
#         np.save(os.path.join(score_dir, f"user_ndcg20_aldi_{save_key}_{g}.npy"), mean_user_g)

#     summary = {
#         "input": save_key,
#         "cfg": cfg,
#         "n_params": n_params,
#         "a_R20_mean": float(np.mean(metrics_a["Recall@20"])),
#         "a_R20_std": float(np.std(metrics_a["Recall@20"], ddof=1)),
#         "a_NDCG20_mean": float(np.mean(metrics_a["NDCG@20"])),
#         "a_NDCG20_std": float(np.std(metrics_a["NDCG@20"], ddof=1)),
#         "b_R20_mean": float(np.mean(metrics_b["Recall@20"])),
#         "b_NDCG20_mean": float(np.mean(metrics_b["NDCG@20"])),
#         "trung_R20_mean": float(np.mean(sub_metrics["trung"]["Recall@20"])),
#         "trung_NDCG20_mean": float(np.mean(sub_metrics["trung"]["NDCG@20"])),
#         "moi_R20_mean": float(np.mean(sub_metrics["moi"]["Recall@20"])),
#         "moi_NDCG20_mean": float(np.mean(sub_metrics["moi"]["NDCG@20"])),
#     }
#     print(
#         f"=> TỔNG KẾT ALDI [{save_key}] (params={n_params:,}):\n"
#         f"   Cold-test (a): NDCG@20 = {summary['a_NDCG20_mean']:.4f} ± {summary['a_NDCG20_std']:.4f} | "
#         f"R@20 = {summary['a_R20_mean']:.4f} ± {summary['a_R20_std']:.4f}\n"
#         f"   All-items (b): NDCG@20 = {summary['b_NDCG20_mean']:.4f} | R@20 = {summary['b_R20_mean']:.4f}\n"
#         f"   Nhóm TRÙNG / MỚI: R@20 = {summary['trung_R20_mean']:.4f} / {summary['moi_R20_mean']:.4f} | "
#         f"NDCG@20 = {summary['trung_NDCG20_mean']:.4f} / {summary['moi_NDCG20_mean']:.4f}\n"
#     )
#     return summary


# # Cấu hình đã chốt ở lần chạy trước của 'raw' để kiểm tra tái lập (KHÔNG chọn lại nếu lệch nhẹ)
# LOCKED_RAW_CFG = {
#     "lr": 0.005,
#     "alpha": 0.9,
#     "beta": 0.01,
#     "gamma": 0.1,
#     "reg": 0.001,
#     "tws": 0,
# }


# def tune_and_run_aldi(
#     variants=("raw", "llm", "fused"),
#     run_erq3=True,
#     run_w6_robustness=True,
# ):
#     out_dir = os.path.join(ad.output_dir, "results_aldi")
#     os.makedirs(out_dir, exist_ok=True)
#     summary_path = os.path.join(out_dir, "aldi_summary.json")
#     all_summaries = {}
#     best_cfgs = {}
#     top3_cfgs = {}

#     # Chạy thực tế cấu hình DEFAULT (seed=42, raw) để kiểm tra tái lập (kỳ vọng 0.0773 tại epoch 5)
#     print("=== KIỂM TRA CẤU HÌNH MẶC ĐỊNH (DEFAULT, seed=42, raw) ===")
#     raw_content_t = torch.tensor(
#         ad.contents["raw"], dtype=torch.float32, device=DEV
#     )
#     _, def_val_ndcg, def_best_ep = train_aldi(raw_content_t, DEFAULT, seed=42)
#     print(
#         f"ALDI chính thức cấu hình DEFAULT (raw, seed=42): "
#         f"cold-val NDCG@20 = {def_val_ndcg:.4f} (tại epoch {def_best_ep}) "
#         f"[Kỳ vọng: 0.0773 tại epoch 5]\n"
#     )

#     # Tinh chỉnh 20 cấu hình (seed=42) cho mọi đầu vào và LƯU LOG trước khi sắp xếp (Phụ lục A.3)
#     for inp in variants:
#         print(f"=== TINH CHỈNH 20 CẤU HÌNH CHO ALDI [{inp}] (seed=42) ===")
#         content_t = torch.tensor(ad.contents[inp], dtype=torch.float32, device=DEV)
#         trial_records = []
#         for idx, cfg in enumerate(trials, 1):
#             _, val_ndcg, best_ep = train_aldi(content_t, cfg, seed=42)
#             trial_records.append((val_ndcg, best_ep, cfg))
#             print(
#                 f"  Trial {idx:02d}/20 | val_NDCG@20={val_ndcg:.4f}"
#                 f" (ep={best_ep:02d}) | {cfg}"
#             )

#         # Lưu log 20 cấu hình trước khi sắp xếp theo đúng Phụ lục A.3
#         json.dump(
#             [
#                 dict(trial=i + 1, val_ndcg20=v, best_ep=e, cfg=c)
#                 for i, (v, e, c) in enumerate(trial_records)
#             ],
#             open(os.path.join(out_dir, f"trials_{inp}.json"), "w"),
#             indent=1,
#         )

#         trial_records.sort(key=lambda x: x[0], reverse=True)
#         best_val, best_ep, best_cfg = trial_records[0]

#         if inp == "raw":
#             print(
#                 f"[Kiểm tra tái lập raw] Best val_NDCG@20={best_val:.4f} tại"
#                 f" ep={best_ep} (Kỳ vọng: 0.1018 tại ep=25)"
#             )
#             if best_cfg != LOCKED_RAW_CFG:
#                 print(
#                     f"[CẢNH BÁO X3] Cấu hình tốt nhất mới ({best_cfg}) khác lần trước."
#                     f" Giữ nguyên cấu hình đã khóa: {LOCKED_RAW_CFG}"
#                 )
#                 best_cfg = LOCKED_RAW_CFG

#         best_cfgs[inp] = best_cfg
#         top3_cfgs[inp] = [r[2] for r in trial_records[:3]]
#         print(
#             f"-> Cấu hình tốt nhất [{inp}]: val_NDCG@20={best_val:.4f}"
#             f" (ep={best_ep}) | {best_cfg}"
#         )

#         all_summaries[inp] = evaluate_5_seeds(inp, best_cfg)
#         with open(summary_path, "w", encoding="utf-8") as f:
#             json.dump(all_summaries, f, indent=2, ensure_ascii=False)

#     # Hồ sơ 3 chiều dùng cấu hình của 5 chiều tương ứng (không tune lại)
#     if run_erq3:
#         for inp_3d, base_5d in (("llm_3d", "llm"), ("fused_3d", "fused")):
#             if inp_3d in ad.contents and base_5d in best_cfgs:
#                 print(f"=== CHẠY ALDI [{inp_3d}] DÙNG CẤU HÌNH CỦA [{base_5d}] ===")
#                 all_summaries[inp_3d] = evaluate_5_seeds(inp_3d, best_cfgs[base_5d])
#                 with open(summary_path, "w", encoding="utf-8") as f:
#                     json.dump(all_summaries, f, indent=2, ensure_ascii=False)

#     # Phân tích độ bền chọn cấu hình bằng trung bình 3 seed trên top-3 trials và lưu log JSON (Phụ lục A.3)
#     if run_w6_robustness:
#         for inp in variants:
#             print(f"=== PHÂN TÍCH ĐỘ BỀN CHỌN 3 SEED CHO ALDI [{inp}] ===")
#             content_t = torch.tensor(
#             ad.contents[inp], dtype=torch.float32, device=DEV
#             )
#             best_3s_val, best_3s_cfg = -1.0, None
#             rob_log = []
#             for cfg in top3_cfgs[inp]:
#                 vals = [train_aldi(content_t, cfg, seed=s)[1] for s in ROBUST_VAL_SEEDS]
#                 m_val = float(np.mean(vals))
#                 rob_log.append(dict(cfg=cfg, val_3seed=vals, mean=m_val))
#                 print(f"  Top-3 candidate {cfg} -> 3-seed cold-val mean = {m_val:.4f}")
#                 if m_val > best_3s_val:
#                     best_3s_val, best_3s_cfg = m_val, cfg

#             json.dump(
#                 rob_log,
#                 open(os.path.join(out_dir, f"robust3s_{inp}.json"), "w"),
#                 indent=1,
#             )
#             print(
#                 f"-> Cấu hình chọn theo 3 seed [{inp}]: {best_3s_cfg}"
#                 f" (val={best_3s_val:.4f})"
#             )
#             all_summaries[f"{inp}_robust3s"] = evaluate_5_seeds(
#                 inp, best_3s_cfg, tag_suffix="_robust3s"
#             )
#             with open(summary_path, "w", encoding="utf-8") as f:
#                 json.dump(all_summaries, f, indent=2, ensure_ascii=False)

# if __name__ == "__main__":
#     tune_and_run_aldi(
#         variants=("raw", "llm", "fused"),
#         run_erq3=True,
#         run_w6_robustness=True,  # Đặt True khi chạy phân tích độ bền 
#     )

In [13]:
# !python run_aldi.py